In [1]:
import json
from pathlib import Path

import torch
from tqdm.auto import tqdm

from load_dataset import (
    build_dataset,
    make_torch_dataset,
)
from metric_torch import bce_dice_loss, dice_coef
from swin_unetr_mc import build_swin_unetr_mc

/mnt/d/data/YSH/data/2026 Senior Spring/00 CSMC324 Artificial Intelligence/final/CMSC324-FinalProject/final_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1777256784.751101  451930 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777256786.505178  451930 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777256841.732387  451930 port.cc:153] oneDNN c

# Load hyperparameter

In [ ]:
with open("./hparams.json", "r") as f:
    config = json.load(f)
    swin_cfg = config["swin_transformer"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("Hparams:", swin_cfg["hparam_grid"]) # TODO: will change once the json structure is updated

Using device: cpu
Resolved hparams: {'patch_size': (64, 64, 64), 'batch_size': 1, 'dropout_rate': 0.2, 'learning_rate': 0.001, 'n_epochs': 5, 'optimizer': 'Adam', 'loss_function': 'bce_dice', 'attn_drop_rate': 0.2, 'dropout_path_rate': 0.2, 'model_out': 'swin_unetr_mc_best.pth', 'feature_size': 24}


# Build datasets

In [3]:
x_train, y_train = build_dataset(training=True, n_train=3)
x_test, y_test = build_dataset(training=False, n_test=3)

# TODO: Replace this temporary test-as-validation setup with a proper validation split.
train_loader = make_torch_dataset(x_train, y_train, training=True)
val_loader = make_torch_dataset(x_test, y_test, training=False)

print("Train samples:", len(x_train), "Val samples:", len(x_test))
print("Train batches:", len(train_loader), "Val batches:", len(val_loader))

  Loading 1/3: BraTS-PED-00001-000
  Loading 2/3: BraTS-PED-00003-000
  Loading 3/3: BraTS-PED-00004-000
  Loading 1/3: BraTS-PED-00002-000
  Loading 2/3: BraTS-PED-00008-000
  Loading 3/3: BraTS-PED-00009-000
Train samples: 24 Val samples: 24
Train batches: 24 Val batches: 24


# Configure model

## PyTorch Training With External Model Import

In [ ]:
model = build_swin_unetr_mc(
    input_shape=(*hparams["patch_size"], 4),
    out_channels=1,
    feature_size=hparams["feature_size"], # TODO: what does this hparameter do?
    drop_rate=hparams["dropout_rate"],
    attn_drop_rate=hparams["attn_drop_rate"],
    dropout_path_rate=hparams["dropout_path_rate"],
    force_mc_dropout=True,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=hparams["learning_rate"])

model_out = Path(hparams["model_out"])

best_val_dice = float("-inf")
history = {"train_loss": [], "train_dice": [], "val_loss": [], "val_dice": []}

print(model.__class__.__name__)
print("Checkpoint path:", model_out)

SwinUNETRWithMCDropout
Checkpoint path: swin_unetr_mc_best.pth


# Train model

In [6]:
for epoch in range(1, hparams["n_epochs"] + 1):
    model.train()
    train_losses = []
    train_dices = []

    for xb, yb in tqdm(train_loader, desc=f"Epoch {epoch} train", leave=False):
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        preds = torch.sigmoid(model(xb))
        loss = bce_dice_loss(yb, preds)
        loss.backward()
        optimizer.step()

        train_losses.append(float(loss.detach().cpu()))
        train_dices.append(float(dice_coef(yb.detach(), preds.detach()).cpu()))

    model.eval()
    val_losses = []
    val_dices = []
    with torch.no_grad():
        for xb, yb in tqdm(val_loader, desc=f"Epoch {epoch} val", leave=False):
            xb, yb = xb.to(device), yb.to(device)
            preds = torch.sigmoid(model(xb))
            loss = bce_dice_loss(yb, preds)
            val_losses.append(float(loss.cpu()))
            val_dices.append(float(dice_coef(yb, preds).cpu()))

    mean_train_loss = sum(train_losses) / max(len(train_losses), 1)
    mean_train_dice = sum(train_dices) / max(len(train_dices), 1)
    mean_val_loss = sum(val_losses) / max(len(val_losses), 1)
    mean_val_dice = sum(val_dices) / max(len(val_dices), 1)

    history["train_loss"].append(mean_train_loss)
    history["train_dice"].append(mean_train_dice)
    history["val_loss"].append(mean_val_loss)
    history["val_dice"].append(mean_val_dice)

    if mean_val_dice > best_val_dice:
        best_val_dice = mean_val_dice
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "hparams": hparams,
                "best_val_dice": best_val_dice,
            },
            model_out,
        )

    print(
        f"Epoch {epoch}/{hparams['n_epochs']} | "
        f"train_loss={mean_train_loss:.4f} train_dice={mean_train_dice:.4f} | "
        f"val_loss={mean_val_loss:.4f} val_dice={mean_val_dice:.4f}"
    )

Epoch 1 train:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 1/5 | train_loss=0.8341 train_dice=0.6194 | val_loss=0.9005 val_dice=0.5678


Epoch 2/5 | train_loss=0.7338 train_dice=0.6859 | val_loss=0.8314 val_dice=0.5800


Epoch 3/5 | train_loss=0.6406 train_dice=0.7083 | val_loss=0.8644 val_dice=0.5237


Epoch 4/5 | train_loss=0.6313 train_dice=0.6967 | val_loss=0.8052 val_dice=0.5973


Epoch 5/5 | train_loss=0.5638 train_dice=0.7271 | val_loss=0.8338 val_dice=0.5379


# Check the result

In [10]:
print(f"Best validation Dice: {best_val_dice:.4f}")
print(f"Saved checkpoint: {model_out}")

checkpoint = torch.load(model_out, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

with torch.no_grad():
    sample_x, sample_y = next(iter(val_loader))
    sample_x, sample_y = sample_x.to(device), sample_y.to(device)
    sample_x = sample_x.to(device)

    mc_preds = [torch.sigmoid(model(sample_x[:1])) for _ in tqdm(range(100))]
    mc_stack = torch.stack(mc_preds, dim=0)
    mc_variance = float(mc_stack.var(dim=0).mean().cpu())

print(f"MC dropout variance mean (single sample): {mc_variance:.6f}")

Best validation Dice: 0.5973
Saved checkpoint: swin_unetr_mc_best.pth


100%|██████████| 100/100 [02:21<00:00,  1.42s/it]

MC dropout variance mean (single sample): 0.001120
